# PR9: Clustering Techniques (CO4)

**Aim:** Implement K-Means and Hierarchical Clustering techniques. Use the Elbow Method and Dendrogram to determine the optimal number of clusters and analyze the generated clusters using a real-world dataset.

**Dataset:** Iris Dataset

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

import warnings
warnings.filterwarnings('ignore')

## 1. Load the Dataset

In [ ]:
# Load Iris dataset
iris = load_iris()

df = pd.DataFrame(iris.data, columns=iris.feature_names)

print('Dataset Shape:', df.shape)
df.head()

## 2. Explore the Dataset

In [ ]:
print('Dataset Information:')
df.info()

print('\nStatistical Summary:')
display(df.describe())

print('\nMissing Values:')
print(df.isnull().sum())

## 3. Standardize the Features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

X_scaled = pd.DataFrame(X_scaled, columns=df.columns)
X_scaled.head()

## 4. K-Means Clustering – Elbow Method

The Elbow Method uses Within-Cluster Sum of Squares (WCSS/Inertia) to identify a suitable number of clusters.

In [ ]:
wcss = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method for Optimal Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS / Inertia')
plt.xticks(range(1, 11))
plt.grid(True)
plt.show()

**Observation:** The elbow is generally observed around **K = 3**, so three clusters are selected for K-Means.

## 5. Apply K-Means Clustering

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

df_kmeans = df.copy()
df_kmeans['KMeans_Cluster'] = kmeans_labels

display(df_kmeans.head())

## 6. Display K-Means Cluster Centers

In [ ]:
centers = pd.DataFrame(kmeans.cluster_centers_, columns=df.columns)
display(centers)

## 7. Visualize K-Means Clusters

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_scaled.iloc[:, 0], X_scaled.iloc[:, 1], c=kmeans_labels, cmap='viridis', s=60)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker='X', s=200, color='black', label='Centroids')
plt.xlabel('Sepal Length (Standardized)')
plt.ylabel('Sepal Width (Standardized)')
plt.title('K-Means Clustering')
plt.legend()
plt.show()

## 8. K-Means Cluster Sizes and Silhouette Score

In [ ]:
cluster_counts = pd.Series(kmeans_labels).value_counts().sort_index()
silhouette_kmeans = silhouette_score(X_scaled, kmeans_labels)

print('K-Means Cluster Sizes:')
print(cluster_counts)
print(f'\nK-Means Silhouette Score: {silhouette_kmeans:.4f}')

# Part B: Hierarchical Clustering

## 9. Generate Linkage Matrix

In [ ]:
linked = linkage(X_scaled, method='ward')
print('Linkage Matrix Shape:', linked.shape)

## 10. Plot Dendrogram

In [ ]:
plt.figure(figsize=(12, 6))
dendrogram(linked, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=10,
           show_contracted=True)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Data Points / Clusters')
plt.ylabel('Distance')
plt.show()

**Observation:** The dendrogram supports selecting approximately **3 clusters** for the Iris dataset.

## 11. Apply Agglomerative Hierarchical Clustering

In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=3, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_scaled)

df_hierarchical = df.copy()
df_hierarchical['Hierarchical_Cluster'] = hierarchical_labels

display(df_hierarchical.head())

## 12. Visualize Hierarchical Clusters

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_scaled.iloc[:, 0], X_scaled.iloc[:, 1],
            c=hierarchical_labels, cmap='plasma', s=60)
plt.xlabel('Sepal Length (Standardized)')
plt.ylabel('Sepal Width (Standardized)')
plt.title('Hierarchical Clustering')
plt.show()

## 13. Hierarchical Cluster Sizes and Silhouette Score

In [ ]:
hierarchical_counts = pd.Series(hierarchical_labels).value_counts().sort_index()
silhouette_hierarchical = silhouette_score(X_scaled, hierarchical_labels)

print('Hierarchical Cluster Sizes:')
print(hierarchical_counts)
print(f'\nHierarchical Silhouette Score: {silhouette_hierarchical:.4f}')

## 14. Cluster-Wise Analysis

In [ ]:
print('K-Means Cluster Analysis:')
display(df_kmeans.groupby('KMeans_Cluster').mean(numeric_only=True))

print('Hierarchical Cluster Analysis:')
display(df_hierarchical.groupby('Hierarchical_Cluster').mean(numeric_only=True))

## 15. Compare K-Means and Hierarchical Clustering

In [ ]:
comparison = pd.DataFrame({
    'Algorithm': ['K-Means', 'Hierarchical Clustering'],
    'Number of Clusters': [3, 3],
    'Silhouette Score': [silhouette_kmeans, silhouette_hierarchical]
})

display(comparison)

## 16. Final Cluster Assignment

In [ ]:
final_df = df.copy()
final_df['KMeans_Cluster'] = kmeans_labels
final_df['Hierarchical_Cluster'] = hierarchical_labels

display(final_df.head(20))

# Conclusion

K-Means and Hierarchical Clustering techniques were successfully implemented on the Iris dataset. The Elbow Method indicated that three clusters provide a suitable grouping, while the Dendrogram also supported the selection of three clusters. K-Means assigned observations into three groups based on feature similarity, and Agglomerative Hierarchical Clustering produced comparable groups. The Silhouette Score was used to evaluate clustering quality. Overall, both techniques successfully identified meaningful patterns and natural groupings in the dataset.